# SQL Wait Time Analysis

This notebook uses SQL to analyze the cleaned Disney Animal Kingdom wait-time dataset. The goal is to answer business questions about attraction wait-time pressure, high-wait risk, and time-based guest experience patterns using SQL queries.

The cleaned dataset was created in the Python notebook `01_data_cleaning_and_wait_time_analysis.ipynb`.

In [1]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

Saving animal_kingdom_wait_times_cleaned.csv to animal_kingdom_wait_times_cleaned.csv


In [2]:
wait_times = pd.read_csv("animal_kingdom_wait_times_cleaned.csv")

wait_times.head()

,date,datetime,posted_wait_minutes,actual_wait_minutes,source_file,attraction_name,posted_wait_invalid,actual_wait_invalid,posted_wait_minutes_clean,actual_wait_minutes_clean,year,month,month_name,day_of_week,hour_of_day,date_only,wait_time_gap,is_high_wait,wait_time_category
0,2012-01-01,2012-01-01 10:28:00,0.0,NaN,AK01.csv,Kilimanjaro Safaris,False,False,0.0,NaN,2012,1,January,Sunday,10,2012-01-01,NaN,0,0-15 min
1,2012-01-01,2012-01-01 11:48:00,5.0,NaN,AK01.csv,Kilimanjaro Safaris,False,False,5.0,NaN,2012,1,January,Sunday,11,2012-01-01,NaN,0,0-15 min
2,2012-01-01,2012-01-01 14:21:00,5.0,NaN,AK01.csv,Kilimanjaro Safaris,False,False,5.0,NaN,2012,1,January,Sunday,14,2012-01-01,NaN,0,0-15 min
3,2012-01-01,2012-01-01 16:40:00,5.0,NaN,AK01.csv,Kilimanjaro Safaris,False,False,5.0,NaN,2012,1,January,Sunday,16,2012-01-01,NaN,0,0-15 min
4,2012-01-01,2012-01-01 17:39:00,0.0,NaN,AK01.csv,Kilimanjaro Safaris,False,False,0.0,NaN,2012,1,January,Sunday,17,2012-01-01,NaN,0,0-15 min


## 1. Create SQLite Database

In this section, I load the cleaned Animal Kingdom wait-time dataset into a SQLite database. This allows the project to demonstrate SQL querying skills using the same cleaned dataset from the Python analysis.

In [3]:
import sqlite3

conn = sqlite3.connect("animal_kingdom_wait_times.db")

wait_times.to_sql(
    "wait_times",
    conn,
    if_exists="replace",
    index=False
)

1905495

In [4]:
# Quick test to make sure the SQLite connection worked
query = """
SELECT *
FROM wait_times
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,date,datetime,posted_wait_minutes,actual_wait_minutes,source_file,attraction_name,posted_wait_invalid,actual_wait_invalid,posted_wait_minutes_clean,actual_wait_minutes_clean,year,month,month_name,day_of_week,hour_of_day,date_only,wait_time_gap,is_high_wait,wait_time_category
0,2012-01-01,2012-01-01 10:28:00,0.0,None,AK01.csv,Kilimanjaro Safaris,0,0,0.0,None,2012,1,January,Sunday,10,2012-01-01,None,0,0-15 min
1,2012-01-01,2012-01-01 11:48:00,5.0,None,AK01.csv,Kilimanjaro Safaris,0,0,5.0,None,2012,1,January,Sunday,11,2012-01-01,None,0,0-15 min
2,2012-01-01,2012-01-01 14:21:00,5.0,None,AK01.csv,Kilimanjaro Safaris,0,0,5.0,None,2012,1,January,Sunday,14,2012-01-01,None,0,0-15 min
3,2012-01-01,2012-01-01 16:40:00,5.0,None,AK01.csv,Kilimanjaro Safaris,0,0,5.0,None,2012,1,January,Sunday,16,2012-01-01,None,0,0-15 min
4,2012-01-01,2012-01-01 17:39:00,0.0,None,AK01.csv,Kilimanjaro Safaris,0,0,0.0,None,2012,1,January,Sunday,17,2012-01-01,None,0,0-15 min


### 1.1 SQLite Connection Test

The test query successfully returned the first five records from the `wait_times` table. This confirms that the cleaned dataset was loaded into SQLite correctly and is ready for SQL-based analysis.

In [5]:
# Testing table structure

query = """
PRAGMA table_info(wait_times);
"""

pd.read_sql_query(query, conn)

,cid,name,type,notnull,dflt_value,pk
0,0,date,TEXT,0,None,0
1,1,datetime,TEXT,0,None,0
2,2,posted_wait_minutes,REAL,0,None,0
3,3,actual_wait_minutes,REAL,0,None,0
4,4,source_file,TEXT,0,None,0
5,5,attraction_name,TEXT,0,None,0
6,6,posted_wait_invalid,INTEGER,0,None,0
7,7,actual_wait_invalid,INTEGER,0,None,0
8,8,posted_wait_minutes_clean,REAL,0,None,0
9,9,actual_wait_minutes_clean,REAL,0,None,0


### 1.2 SQL Table Structure Summary

The table structure confirms that the cleaned Animal Kingdom wait-time dataset was successfully loaded into SQLite with 19 columns. The SQL table includes the original wait-time fields, cleaned wait-time fields, attraction names, time-based fields, and guest experience KPI fields created in the Python notebook.

This table is ready for SQL analysis focused on attraction-level wait-time pressure, high-wait risk, and time-based guest experience patterns.

## 2. Overall Wait-Time KPI Summary

This section uses SQL to calculate high-level wait-time KPIs for the cleaned Disney Animal Kingdom wait-time dataset. These metrics provide an overview of valid posted wait-time records, average posted wait time, maximum posted wait time, and the overall share of high-wait records.

In [6]:
query = """
SELECT
    COUNT(*) AS valid_posted_wait_records,
    ROUND(AVG(posted_wait_minutes_clean), 2) AS avg_posted_wait,
    ROUND(MIN(posted_wait_minutes_clean), 2) AS min_posted_wait,
    ROUND(MAX(posted_wait_minutes_clean), 2) AS max_posted_wait,
    SUM(is_high_wait) AS high_wait_records,
    ROUND(AVG(is_high_wait) * 100, 2) AS high_wait_rate_pct
FROM wait_times
WHERE posted_wait_minutes_clean IS NOT NULL;
"""

overall_kpi_summary = pd.read_sql_query(query, conn)
overall_kpi_summary

,valid_posted_wait_records,avg_posted_wait,min_posted_wait,max_posted_wait,high_wait_records,high_wait_rate_pct
0,1758656,30.52,0.0,390.0,388915,22.11


### Interpretation

The SQL KPI summary shows that the cleaned dataset contains 1,758,656 valid posted wait-time records. Across all Animal Kingdom attractions, the average posted wait time was approximately 30.52 minutes.

The dataset includes 388,915 high-wait records, meaning 22.11% of valid posted wait-time records were 45 minutes or longer. This shows that while most posted waits were below the high-wait threshold, long waits still represented a meaningful share of guest experience risk.

The maximum posted wait time was 390 minutes, suggesting that some attractions experienced extremely high posted waits during peak demand periods.

## 3. Attraction-Level Wait-Time Analysis

This section uses SQL to compare wait-time performance across Animal Kingdom attractions. The goal is to identify which attractions had the highest average posted waits, highest maximum waits, and largest number of valid wait-time records.

### 3.1 Attraction Wait-Time Summary

This query summarizes wait-time performance by attraction. It calculates the number of valid posted wait-time records, average posted wait time, median-level context through minimum and maximum waits, and the number of high-wait records.

In [7]:
query = """
SELECT
    attraction_name,
    COUNT(posted_wait_minutes_clean) AS valid_posted_wait_records,
    ROUND(AVG(posted_wait_minutes_clean), 2) AS avg_posted_wait,
    ROUND(MIN(posted_wait_minutes_clean), 2) AS min_posted_wait,
    ROUND(MAX(posted_wait_minutes_clean), 2) AS max_posted_wait,
    SUM(is_high_wait) AS high_wait_records,
    ROUND(AVG(is_high_wait) * 100, 2) AS high_wait_rate_pct
FROM wait_times
WHERE posted_wait_minutes_clean IS NOT NULL
GROUP BY attraction_name
ORDER BY avg_posted_wait DESC;
"""

attraction_sql_summary = pd.read_sql_query(query, conn)
attraction_sql_summary

,attraction_name,valid_posted_wait_records,avg_posted_wait,min_posted_wait,max_posted_wait,high_wait_records,high_wait_rate_pct
0,Avatar Flight of Passage,78581,136.03,0.0,390.0,77949,99.20
1,Na'vi River Journey,76758,75.29,0.0,225.0,65325,85.11
2,DINOSAUR,217285,34.81,0.0,195.0,72908,33.55
3,Kali River Rapids,183719,33.50,0.0,210.0,55798,30.37
4,Expedition Everest,224460,31.03,0.0,180.0,56892,25.35
5,Primeval Whirl,204465,24.86,0.0,300.0,31320,15.32
6,Adventurers Outpost,147933,23.84,0.0,205.0,14795,10.00
7,It's Tough to Be a Bug,214691,19.89,0.0,120.0,12491,5.82
8,Finding Nemo - The Musical,2921,18.37,0.0,100.0,192,6.57
9,Kilimanjaro Safaris,213522,9.77,0.0,160.0,1129,0.53


### Interpretation

The SQL attraction-level summary shows that Avatar Flight of Passage had the highest average posted wait time at approximately 136 minutes. It also had the highest high-wait rate at 99.20%, meaning nearly every valid posted wait-time record was 45 minutes or longer.

Na'vi River Journey had the second-highest average posted wait time at approximately 75 minutes and a high-wait rate of 85.11%. Together, these two attractions stand out as the clearest wait-time pressure points in the dataset.

After the two Pandora attractions, average posted waits dropped substantially. DINOSAUR, Kali River Rapids, and Expedition Everest had moderate average posted waits between approximately 31 and 35 minutes. This suggests that long-wait guest experience risk was concentrated most heavily around a small group of high-demand attractions.

### 3.2 Top Attractions by High-Wait Records

This query ranks attractions by the total number of high-wait records. A high-wait record is defined as a valid posted wait-time record of 45 minutes or more. This helps identify attractions that contributed the largest volume of long-wait observations.

In [8]:
query = """
SELECT
    attraction_name,
    COUNT(posted_wait_minutes_clean) AS valid_posted_wait_records,
    SUM(is_high_wait) AS high_wait_records,
    ROUND(AVG(is_high_wait) * 100, 2) AS high_wait_rate_pct,
    ROUND(AVG(posted_wait_minutes_clean), 2) AS avg_posted_wait
FROM wait_times
WHERE posted_wait_minutes_clean IS NOT NULL
GROUP BY attraction_name
ORDER BY high_wait_records DESC;
"""

top_high_wait_sql = pd.read_sql_query(query, conn)
top_high_wait_sql

,attraction_name,valid_posted_wait_records,high_wait_records,high_wait_rate_pct,avg_posted_wait
0,Avatar Flight of Passage,78581,77949,99.20,136.03
1,DINOSAUR,217285,72908,33.55,34.81
2,Na'vi River Journey,76758,65325,85.11,75.29
3,Expedition Everest,224460,56892,25.35,31.03
4,Kali River Rapids,183719,55798,30.37,33.50
5,Primeval Whirl,204465,31320,15.32,24.86
6,Adventurers Outpost,147933,14795,10.00,23.84
7,It's Tough to Be a Bug,214691,12491,5.82,19.89
8,Kilimanjaro Safaris,213522,1129,0.53,9.77
9,Finding Nemo - The Musical,2921,192,6.57,18.37


### Interpretation

The SQL high-wait record summary shows that Avatar Flight of Passage had the highest total number of high-wait records, with 77,949 valid posted wait-time records of 45 minutes or more.

DINOSAUR ranked second by total high-wait records, even though its average posted wait time and high-wait rate were much lower than Na'vi River Journey. This suggests that DINOSAUR created meaningful guest experience pressure because it had a large number of valid wait-time observations and a substantial volume of long-wait records.

Na'vi River Journey, Expedition Everest, and Kali River Rapids also contributed a large number of high-wait records. Together, these attractions represent the main sources of long-wait volume in the dataset.

## 4. Time-Based Wait-Time Analysis

This section uses SQL to analyze how posted wait times changed across different time periods. The goal is to identify when guests were most likely to experience higher wait-time pressure.

### 4.1 Average Posted Wait Time by Hour of Day

This query calculates average posted wait time and high-wait rate by hour of day. The analysis is filtered to typical operating hours to reduce the impact of unusual late-night or early-morning records.

In [9]:
query = """
SELECT
    hour_of_day,
    COUNT(posted_wait_minutes_clean) AS valid_posted_wait_records,
    ROUND(AVG(posted_wait_minutes_clean), 2) AS avg_posted_wait,
    SUM(is_high_wait) AS high_wait_records,
    ROUND(AVG(is_high_wait) * 100, 2) AS high_wait_rate_pct
FROM wait_times
WHERE posted_wait_minutes_clean IS NOT NULL
  AND hour_of_day BETWEEN 6 AND 22
GROUP BY hour_of_day
ORDER BY hour_of_day;
"""

hourly_sql_summary = pd.read_sql_query(query, conn)
hourly_sql_summary

,hour_of_day,valid_posted_wait_records,avg_posted_wait,high_wait_records,high_wait_rate_pct
0,6,444,17.31,75,16.89
1,7,11627,16.99,1314,11.30
2,8,66288,17.23,5821,8.78
3,9,147375,21.01,15115,10.26
4,10,157263,29.79,30005,19.08
5,11,157756,35.71,44774,28.38
6,12,157105,36.75,46539,29.62
7,13,156281,35.67,44910,28.74
8,14,156156,35.38,44333,28.39
9,15,154510,33.52,40266,26.06


### Interpretation

The SQL hourly wait-time summary shows that average posted wait times were lowest during the early operating hours, especially from 7 AM to 8 AM. Wait times began rising around 9 AM and increased sharply by 10 AM.

The highest average posted wait time occurred at 12 PM, with an average posted wait of 36.75 minutes. The high-wait rate also peaked at 12 PM, when 29.62% of valid posted wait-time records were 45 minutes or longer.

After midday, both average posted wait times and high-wait rates gradually declined through the afternoon and evening. This suggests that late morning through early afternoon was the highest-pressure period for guest wait-time experience at Animal Kingdom.

### 4.2 Average Posted Wait Time by Day of Week

This query calculates average posted wait time and high-wait rate by day of week. The goal is to determine whether certain days of the week created higher wait-time pressure for guests.

In [10]:
query = """
SELECT
    day_of_week,
    COUNT(posted_wait_minutes_clean) AS valid_posted_wait_records,
    ROUND(AVG(posted_wait_minutes_clean), 2) AS avg_posted_wait,
    SUM(is_high_wait) AS high_wait_records,
    ROUND(AVG(is_high_wait) * 100, 2) AS high_wait_rate_pct
FROM wait_times
WHERE posted_wait_minutes_clean IS NOT NULL
GROUP BY day_of_week
ORDER BY
    CASE day_of_week
        WHEN 'Monday' THEN 1
        WHEN 'Tuesday' THEN 2
        WHEN 'Wednesday' THEN 3
        WHEN 'Thursday' THEN 4
        WHEN 'Friday' THEN 5
        WHEN 'Saturday' THEN 6
        WHEN 'Sunday' THEN 7
    END;
"""

day_sql_summary = pd.read_sql_query(query, conn)
day_sql_summary

,day_of_week,valid_posted_wait_records,avg_posted_wait,high_wait_records,high_wait_rate_pct
0,Monday,258898,31.47,61344,23.69
1,Tuesday,245373,30.43,53949,21.99
2,Wednesday,246506,28.55,49680,20.15
3,Thursday,242138,30.31,52631,21.74
4,Friday,244933,30.61,53186,21.71
5,Saturday,263702,31.82,61321,23.25
6,Sunday,257106,30.28,56804,22.09


### Interpretation

The SQL day-of-week summary shows that average posted wait times were fairly consistent across the week. Saturday had the highest average posted wait time at 31.82 minutes, followed closely by Monday at 31.47 minutes. Wednesday had the lowest average posted wait time at 28.55 minutes.

The high-wait rate followed a similar pattern, with Monday and Saturday showing the highest rates and Wednesday showing the lowest. However, the differences between days were relatively small. This suggests that day of week was not as strong of a driver of guest wait-time pressure as attraction type or hour of day..

### 4.3 Average Posted Wait Time by Month

This query calculates average posted wait time and high-wait rate by month. The goal is to identify seasonal wait-time patterns and determine which months created the highest guest experience pressure.

In [11]:
query = """
SELECT
    month,
    month_name,
    COUNT(posted_wait_minutes_clean) AS valid_posted_wait_records,
    ROUND(AVG(posted_wait_minutes_clean), 2) AS avg_posted_wait,
    SUM(is_high_wait) AS high_wait_records,
    ROUND(AVG(is_high_wait) * 100, 2) AS high_wait_rate_pct
FROM wait_times
WHERE posted_wait_minutes_clean IS NOT NULL
GROUP BY month, month_name
ORDER BY month;
"""

month_sql_summary = pd.read_sql_query(query, conn)
month_sql_summary

,month,month_name,valid_posted_wait_records,avg_posted_wait,high_wait_records,high_wait_rate_pct
0,1,January,137696,33.11,31970,23.22
1,2,February,133897,32.58,31913,23.83
2,3,March,135578,31.76,33064,24.39
3,4,April,135099,28.98,28798,21.32
4,5,May,142114,27.95,29102,20.48
5,6,June,167959,31.63,40292,23.99
6,7,July,159118,31.73,39162,24.61
7,8,August,155469,29.25,32702,21.03
8,9,September,139573,22.19,18374,13.16
9,10,October,150650,29.37,30779,20.43


### Interpretation

The SQL monthly wait-time summary shows a clear seasonal pattern in posted wait-time pressure. December had the highest average posted wait time at 35.55 minutes and the highest high-wait rate at 25.65%. This suggests that December was the strongest monthly pressure point in the dataset.

September had the lowest average posted wait time at 22.19 minutes and the lowest high-wait rate at 13.16%, making it the lowest-pressure month for guest wait times.

Other relatively high-pressure months included January, February, March, June, July, and November. This pattern suggests that wait-time pressure increased during several common vacation and holiday travel periods, while September showed a noticeably lower level of demand.

## 5. SQL Key Findings

This section summarizes the main insights from the SQL analysis. These findings support the broader Python analysis and show how SQL can be used to answer business questions about attraction wait-time pressure and guest experience risk.

### Key Findings

1. **The dataset contained over 1.75 million valid posted wait-time records.**  
   The SQL analysis found 1,758,656 valid posted wait-time records after excluding invalid or missing posted wait values.

2. **Long waits represented a meaningful share of guest experience risk.**  
   Across all valid posted wait-time records, 388,915 records were 45 minutes or longer. This produced an overall high-wait rate of 22.11%.

3. **Avatar Flight of Passage was the strongest wait-time pressure point.**  
   Avatar Flight of Passage had the highest average posted wait time at approximately 136 minutes and the highest high-wait rate at 99.20%.

4. **DINOSAUR created a large volume of high-wait records.**  
   Although DINOSAUR had a lower average posted wait than Na'vi River Journey, it ranked second in total high-wait records. This shows why both rate-based and volume-based metrics are useful.

5. **Wait-time pressure peaked around midday.**  
   The highest average posted wait time and high-wait rate occurred at 12 PM, suggesting that late morning through early afternoon was the most important period for guest experience monitoring.

6. **Day of week was not a major driver of wait-time differences.**  
   Average posted waits were fairly consistent across the week, with only small differences between the highest and lowest days.

7. **December showed the highest monthly wait-time pressure.**  
   December had the highest average posted wait time and highest high-wait rate, while September had the lowest values.

### SQL Analysis Summary

The SQL analysis confirmed the major findings from the Python notebook. Wait-time pressure at Animal Kingdom was concentrated around a small number of high-demand attractions, especially Avatar Flight of Passage and Na'vi River Journey. The analysis also showed that time of day was an important driver of guest experience risk, with wait-time pressure peaking around midday.

By using SQL to calculate attraction-level, hourly, weekly, and monthly KPIs, this notebook demonstrates how relational querying can support business analysis and operational decision-making.